# 14 Command Line Arguments — Advanced

This notebook covers advanced command-line argument handling in Python using `argparse`.

Topics:

- testable CLI design with `main(argv=None)`
- positional arguments
- optional arguments
- flags
- mutually exclusive groups
- choices
- custom type validation
- subcommands
- environment-variable fallback
- reading files safely
- structured output
- advanced testing patterns

The goal is to write command-line programs that are clean, testable, and user-friendly.

In [1]:
import argparse
import contextlib
import csv
import io
import json
import math
import os
import pathlib
import statistics
import tempfile
from dataclasses import dataclass

## Notebook helper: run CLI code without using the real terminal

A well-designed CLI should have a `main(argv=None)` function.

That lets us test it like this:

```python
main(["--verbose", "input.txt"])
```

instead of depending directly on `sys.argv`.

In [2]:
def run_cli(main_func, argv, *, env=None):
    """
    Run a CLI-style main(argv) function and capture stdout, stderr, and exit code.

    Parameters
    ----------
    main_func:
        Function shaped like main(argv=None).
    argv:
        List of command-line arguments, excluding the program name.
    env:
        Optional temporary environment-variable overrides.

    Returns
    -------
    dict with keys: code, stdout, stderr
    """
    out = io.StringIO()
    err = io.StringIO()

    old_env = os.environ.copy()
    if env:
        os.environ.update(env)

    try:
        try:
            with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
                result = main_func(argv)
            code = 0 if result is None else result
        except SystemExit as exc:
            code = exc.code
    finally:
        os.environ.clear()
        os.environ.update(old_env)

    return {
        "code": code,
        "stdout": out.getvalue(),
        "stderr": err.getvalue(),
    }


def show_result(result):
    print("exit code:", result["code"])
    if result["stdout"]:
        print("--- stdout ---")
        print(result["stdout"])
    if result["stderr"]:
        print("--- stderr ---")
        print(result["stderr"])

## Problem 1 — Build a testable CLI skeleton

Create a CLI named `greet` with:

- one positional argument: `name`
- optional argument: `--title`
- flag: `--shout`

Examples:

```bash
python greet.py Ada
python greet.py Ada --title Dr.
python greet.py Ada --title Dr. --shout
```

Expected behavior:

- `Ada` → `Hello, Ada!`
- `Ada --title Dr.` → `Hello, Dr. Ada!`
- `Ada --shout` → uppercase output

In [3]:
def build_parser_problem_1():
    parser = argparse.ArgumentParser(
        prog="greet",
        description="Greet a person from the command line."
    )
    parser.add_argument("name", help="person to greet")
    parser.add_argument("--title", help="optional title, such as Dr. or Prof.")
    parser.add_argument("--shout", action="store_true", help="print greeting in uppercase")
    return parser


def main_problem_1(argv=None):
    parser = build_parser_problem_1()
    args = parser.parse_args(argv)

    display_name = args.name
    if args.title:
        display_name = f"{args.title} {display_name}"

    message = f"Hello, {display_name}!"
    if args.shout:
        message = message.upper()

    print(message)
    return 0


for argv in [
    ["Ada"],
    ["Ada", "--title", "Dr."],
    ["Ada", "--title", "Dr.", "--shout"]
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_1, argv))
    print()

argv: ['Ada']
exit code: 0
--- stdout ---
Hello, Ada!


argv: ['Ada', '--title', 'Dr.']
exit code: 0
--- stdout ---
Hello, Dr. Ada!


argv: ['Ada', '--title', 'Dr.', '--shout']
exit code: 0
--- stdout ---
HELLO, DR. ADA!




### Best-practice notes

The parser construction is separated from the execution logic.

This makes the program easier to test, reuse, and maintain.

## Problem 2 — Use choices for controlled input

Create a CLI named `convert-temp` that converts temperatures.

Arguments:

- positional `value`: a floating-point temperature
- required option `--from-unit`: one of `C`, `F`, `K`
- required option `--to-unit`: one of `C`, `F`, `K`
- optional flag `--round`: round the result to 2 decimal places

Reject unsupported units automatically using `choices`.

In [4]:
def to_celsius(value, unit):
    if unit == "C":
        return value
    if unit == "F":
        return (value - 32) * 5 / 9
    if unit == "K":
        return value - 273.15
    raise ValueError(f"unsupported unit: {unit}")


def from_celsius(value, unit):
    if unit == "C":
        return value
    if unit == "F":
        return value * 9 / 5 + 32
    if unit == "K":
        return value + 273.15
    raise ValueError(f"unsupported unit: {unit}")


def build_parser_problem_2():
    parser = argparse.ArgumentParser(prog="convert-temp")
    parser.add_argument("value", type=float, help="temperature value")
    parser.add_argument("--from-unit", choices=["C", "F", "K"], required=True)
    parser.add_argument("--to-unit", choices=["C", "F", "K"], required=True)
    parser.add_argument("--round", action="store_true", help="round to 2 decimal places")
    return parser


def main_problem_2(argv=None):
    parser = build_parser_problem_2()
    args = parser.parse_args(argv)

    celsius = to_celsius(args.value, args.from_unit)
    result = from_celsius(celsius, args.to_unit)

    if args.round:
        result = round(result, 2)

    print(result)
    return 0


examples = [
    ["100", "--from-unit", "C", "--to-unit", "F"],
    ["32", "--from-unit", "F", "--to-unit", "C", "--round"],
    ["300", "--from-unit", "K", "--to-unit", "C", "--round"],
    ["100", "--from-unit", "X", "--to-unit", "C"]
]

for argv in examples:
    print("argv:", argv)
    show_result(run_cli(main_problem_2, argv))
    print()

argv: ['100', '--from-unit', 'C', '--to-unit', 'F']
exit code: 0
--- stdout ---
212.0


argv: ['32', '--from-unit', 'F', '--to-unit', 'C', '--round']
exit code: 0
--- stdout ---
0.0


argv: ['300', '--from-unit', 'K', '--to-unit', 'C', '--round']
exit code: 0
--- stdout ---
26.85


argv: ['100', '--from-unit', 'X', '--to-unit', 'C']
exit code: 2
--- stderr ---
usage: convert-temp [-h] --from-unit {C,F,K} --to-unit {C,F,K} [--round] value
convert-temp: error: argument --from-unit: invalid choice: 'X' (choose from C, F, K)




### Best-practice notes

Use `choices` when a parameter has a small fixed set of valid values.

This avoids manual validation and gives the user a clear error message.

## Problem 3 — Mutually exclusive flags

Create a CLI named `summarize` that accepts numbers and supports three output modes:

- default: print count, mean, min, and max
- `--quiet`: print only the mean
- `--verbose`: print the original values and all summary statistics

`--quiet` and `--verbose` must not be allowed together.

In [5]:
def build_parser_problem_3():
    parser = argparse.ArgumentParser(prog="summarize")
    parser.add_argument("numbers", nargs="+", type=float, help="one or more numbers")

    group = parser.add_mutually_exclusive_group()
    group.add_argument("-q", "--quiet", action="store_true", help="print only the mean")
    group.add_argument("-v", "--verbose", action="store_true", help="print detailed output")

    return parser


def summarize_numbers(numbers):
    return {
        "count": len(numbers),
        "mean": statistics.mean(numbers),
        "min": min(numbers),
        "max": max(numbers),
    }


def main_problem_3(argv=None):
    parser = build_parser_problem_3()
    args = parser.parse_args(argv)

    stats = summarize_numbers(args.numbers)

    if args.quiet:
        print(stats["mean"])
    elif args.verbose:
        print("values:", args.numbers)
        for key, value in stats.items():
            print(f"{key}: {value}")
    else:
        print(f"count={stats['count']} mean={stats['mean']} min={stats['min']} max={stats['max']}")

    return 0


for argv in [
    ["1", "2", "3", "4"],
    ["--quiet", "1", "2", "3", "4"],
    ["--verbose", "1", "2", "3", "4"],
    ["--quiet", "--verbose", "1", "2"]
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_3, argv))
    print()

argv: ['1', '2', '3', '4']
exit code: 0
--- stdout ---
count=4 mean=2.5 min=1.0 max=4.0


argv: ['--quiet', '1', '2', '3', '4']
exit code: 0
--- stdout ---
2.5


argv: ['--verbose', '1', '2', '3', '4']
exit code: 0
--- stdout ---
values: [1.0, 2.0, 3.0, 4.0]
count: 4
mean: 2.5
min: 1.0
max: 4.0


argv: ['--quiet', '--verbose', '1', '2']
exit code: 2
--- stderr ---
usage: summarize [-h] [-q | -v] numbers [numbers ...]
summarize: error: argument -v/--verbose: not allowed with argument -q/--quiet




### Best-practice notes

`add_mutually_exclusive_group()` is useful when two options represent conflicting behaviors.

Examples:

- `--quiet` vs `--verbose`
- `--dry-run` vs `--execute`
- `--json` vs `--csv`
- `--include` vs `--exclude`

## Problem 4 — Store mode as a single value

Booleans can become messy when options represent modes.

Rewrite the previous problem so that the parsed namespace contains:

```python
args.mode
```

where `mode` is one of:

- `"normal"`
- `"quiet"`
- `"verbose"`

Use `action="store_const"`.

In [6]:
def build_parser_problem_4():
    parser = argparse.ArgumentParser(prog="summarize-mode")
    parser.add_argument("numbers", nargs="+", type=float)

    group = parser.add_mutually_exclusive_group()
    group.add_argument("-q", "--quiet", dest="mode", action="store_const", const="quiet")
    group.add_argument("-v", "--verbose", dest="mode", action="store_const", const="verbose")

    parser.set_defaults(mode="normal")
    return parser


def main_problem_4(argv=None):
    parser = build_parser_problem_4()
    args = parser.parse_args(argv)

    stats = summarize_numbers(args.numbers)

    if args.mode == "quiet":
        print(stats["mean"])
    elif args.mode == "verbose":
        print(json.dumps({"values": args.numbers, "stats": stats}, indent=2))
    else:
        print(stats)

    return 0


for argv in [
    ["10", "20", "30"],
    ["--quiet", "10", "20", "30"],
    ["--verbose", "10", "20", "30"]
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_4, argv))
    print()

argv: ['10', '20', '30']
exit code: 0
--- stdout ---
{'count': 3, 'mean': 20.0, 'min': 10.0, 'max': 30.0}


argv: ['--quiet', '10', '20', '30']
exit code: 0
--- stdout ---
20.0


argv: ['--verbose', '10', '20', '30']
exit code: 0
--- stdout ---
{
  "values": [
    10.0,
    20.0,
    30.0
  ],
  "stats": {
    "count": 3,
    "mean": 20.0,
    "min": 10.0,
    "max": 30.0
  }
}




### Best-practice notes

A single mode variable is often cleaner than several booleans.

Prefer this:

```python
args.mode == "verbose"
```

over this:

```python
args.verbose and not args.quiet and not args.debug
```

## Problem 5 — Custom type validation

Create a CLI named `circle`.

Arguments:

- required option `--radius`
- optional `--format`, choices: `text` or `json`

The radius must be:

- a number
- finite
- strictly positive

Use a custom type function called `positive_finite_float`.

In [7]:
def positive_finite_float(text):
    try:
        value = float(text)
    except ValueError as exc:
        raise argparse.ArgumentTypeError(f"{text!r} is not a valid number") from exc

    if not math.isfinite(value):
        raise argparse.ArgumentTypeError("value must be finite")

    if value <= 0:
        raise argparse.ArgumentTypeError("value must be greater than 0")

    return value


def build_parser_problem_5():
    parser = argparse.ArgumentParser(prog="circle")
    parser.add_argument("--radius", type=positive_finite_float, required=True)
    parser.add_argument("--format", choices=["text", "json"], default="text")
    return parser


def main_problem_5(argv=None):
    parser = build_parser_problem_5()
    args = parser.parse_args(argv)

    area = math.pi * args.radius ** 2
    circumference = 2 * math.pi * args.radius

    if args.format == "json":
        print(json.dumps({
            "radius": args.radius,
            "area": area,
            "circumference": circumference
        }, sort_keys=True))
    else:
        print(f"radius={args.radius}")
        print(f"area={area}")
        print(f"circumference={circumference}")

    return 0


for argv in [
    ["--radius", "3"],
    ["--radius", "3", "--format", "json"],
    ["--radius", "0"],
    ["--radius", "nan"]
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_5, argv))
    print()

argv: ['--radius', '3']
exit code: 0
--- stdout ---
radius=3.0
area=28.274333882308138
circumference=18.84955592153876


argv: ['--radius', '3', '--format', 'json']
exit code: 0
--- stdout ---
{"area": 28.274333882308138, "circumference": 18.84955592153876, "radius": 3.0}


argv: ['--radius', '0']
exit code: 2
--- stderr ---
usage: circle [-h] --radius RADIUS [--format {text,json}]
circle: error: argument --radius: value must be greater than 0


argv: ['--radius', 'nan']
exit code: 2
--- stderr ---
usage: circle [-h] --radius RADIUS [--format {text,json}]
circle: error: argument --radius: value must be finite




### Best-practice notes

Use `argparse.ArgumentTypeError` inside custom type functions.

That lets `argparse` produce standard CLI-style error output.

## Problem 6 — Validate relationships between arguments

Some validation cannot be done by a single argument type.

Create a CLI named `slice-range`.

Arguments:

- `--start`: integer, required
- `--stop`: integer, required
- `--step`: integer, default `1`

Rules:

- `step` cannot be zero
- if `step > 0`, then `start < stop`
- if `step < 0`, then `start > stop`

Use `parser.error(...)` for cross-argument validation.

In [8]:
def build_parser_problem_6():
    parser = argparse.ArgumentParser(prog="slice-range")
    parser.add_argument("--start", type=int, required=True)
    parser.add_argument("--stop", type=int, required=True)
    parser.add_argument("--step", type=int, default=1)
    return parser


def validate_range_args(parser, args):
    if args.step == 0:
        parser.error("--step cannot be zero")

    if args.step > 0 and args.start >= args.stop:
        parser.error("when --step is positive, --start must be less than --stop")

    if args.step < 0 and args.start <= args.stop:
        parser.error("when --step is negative, --start must be greater than --stop")


def main_problem_6(argv=None):
    parser = build_parser_problem_6()
    args = parser.parse_args(argv)
    validate_range_args(parser, args)

    print(list(range(args.start, args.stop, args.step)))
    return 0


for argv in [
    ["--start", "1", "--stop", "5"],
    ["--start", "5", "--stop", "1", "--step", "-1"],
    ["--start", "5", "--stop", "1"],
    ["--start", "1", "--stop", "5", "--step", "0"]
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_6, argv))
    print()

argv: ['--start', '1', '--stop', '5']
exit code: 0
--- stdout ---
[1, 2, 3, 4]


argv: ['--start', '5', '--stop', '1', '--step', '-1']
exit code: 0
--- stdout ---
[5, 4, 3, 2]


argv: ['--start', '5', '--stop', '1']
exit code: 2
--- stderr ---
usage: slice-range [-h] --start START --stop STOP [--step STEP]
slice-range: error: when --step is positive, --start must be less than --stop


argv: ['--start', '1', '--stop', '5', '--step', '0']
exit code: 2
--- stderr ---
usage: slice-range [-h] --start START --stop STOP [--step STEP]
slice-range: error: --step cannot be zero




### Best-practice notes

Use custom `type=` functions for single-argument validation.

Use post-parse validation for relationships between multiple arguments.

## Problem 7 — Read from a file path

Create a CLI named `word-count`.

Arguments:

- positional `path`: path to a text file
- optional `--json`: output JSON instead of text

The program should print:

- number of lines
- number of words
- number of characters

Use `pathlib.Path` instead of plain strings.

In [9]:
def existing_file_path(text):
    path = pathlib.Path(text)
    if not path.exists():
        raise argparse.ArgumentTypeError(f"file does not exist: {text}")
    if not path.is_file():
        raise argparse.ArgumentTypeError(f"not a file: {text}")
    return path


def count_text(text):
    return {
        "lines": len(text.splitlines()),
        "words": len(text.split()),
        "characters": len(text),
    }


def build_parser_problem_7():
    parser = argparse.ArgumentParser(prog="word-count")
    parser.add_argument("path", type=existing_file_path)
    parser.add_argument("--json", action="store_true", help="print JSON output")
    return parser


def main_problem_7(argv=None):
    parser = build_parser_problem_7()
    args = parser.parse_args(argv)

    text = args.path.read_text(encoding="utf-8")
    counts = count_text(text)

    if args.json:
        print(json.dumps(counts, sort_keys=True))
    else:
        for key, value in counts.items():
            print(f"{key}: {value}")

    return 0


with tempfile.TemporaryDirectory() as tmpdir:
    path = pathlib.Path(tmpdir) / "sample.txt"
    path.write_text("hello world\nthis is a test\n", encoding="utf-8")

    for argv in [[str(path)], [str(path), "--json"], [str(path) + ".missing"]]:
        print("argv:", argv)
        show_result(run_cli(main_problem_7, argv))
        print()

argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmp3r31hvv9\\sample.txt']
exit code: 0
--- stdout ---
lines: 2
words: 6
characters: 27


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmp3r31hvv9\\sample.txt', '--json']
exit code: 0
--- stdout ---
{"characters": 27, "lines": 2, "words": 6}


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmp3r31hvv9\\sample.txt.missing']
exit code: 2
--- stderr ---
usage: word-count [-h] [--json] path
word-count: error: argument path: file does not exist: C:\Users\user1\AppData\Local\Temp\tmp3r31hvv9\sample.txt.missing




### Best-practice notes

Using `pathlib.Path` makes file handling cleaner and more portable.

For real production CLIs, consider whether the file should be validated during parsing or opened later so that errors can be handled more flexibly.

## Problem 8 — Environment-variable fallback

Create a CLI named `api-client`.

It needs an API token.

The token can come from:

1. `--token TOKEN`
2. environment variable `API_TOKEN`

Command-line argument takes priority.

If no token is available, produce a clean parser error.

In [10]:
def build_parser_problem_8():
    parser = argparse.ArgumentParser(prog="api-client")
    parser.add_argument("--token", help="API token. If omitted, API_TOKEN is used.")
    parser.add_argument("--endpoint", default="status", choices=["status", "users", "reports"])
    return parser


def main_problem_8(argv=None):
    parser = build_parser_problem_8()
    args = parser.parse_args(argv)

    token = args.token or os.environ.get("API_TOKEN")
    if not token:
        parser.error("API token required: pass --token or set API_TOKEN")

    masked = token[:4] + "..." if len(token) > 4 else "****"
    print(f"endpoint={args.endpoint}")
    print(f"token={masked}")
    return 0


examples = [
    ({}, []),
    ({"API_TOKEN": "env-secret-token"}, []),
    ({"API_TOKEN": "env-secret-token"}, ["--token", "cli-secret-token", "--endpoint", "reports"])
]

for env, argv in examples:
    print("env:", env, "argv:", argv)
    show_result(run_cli(main_problem_8, argv, env=env))
    print()

env: {} argv: []
exit code: 2
--- stderr ---
usage: api-client [-h] [--token TOKEN] [--endpoint {status,users,reports}]
api-client: error: API token required: pass --token or set API_TOKEN


env: {'API_TOKEN': 'env-secret-token'} argv: []
exit code: 0
--- stdout ---
endpoint=status
token=env-...


env: {'API_TOKEN': 'env-secret-token'} argv: ['--token', 'cli-secret-token', '--endpoint', 'reports']
exit code: 0
--- stdout ---
endpoint=reports
token=cli-...




### Best-practice notes

Do not make secrets required command-line arguments when environment variables are more appropriate.

Command-line tokens can appear in shell history or process listings.

## Problem 9 — Subcommands

Create a CLI named `todo` with subcommands:

```bash
todo add "Buy milk"
todo list
todo done 2
```

For this notebook, do not use a real database. Just parse the command and print what would happen.

Subcommands:

- `add TEXT`
- `list --all`
- `done ID`

Use `set_defaults(command=...)` or equivalent parser metadata.

In [11]:
def build_parser_problem_9():
    parser = argparse.ArgumentParser(prog="todo")
    subparsers = parser.add_subparsers(dest="command", required=True)

    add_parser = subparsers.add_parser("add", help="add a new todo item")
    add_parser.add_argument("text", help="todo text")

    list_parser = subparsers.add_parser("list", help="list todo items")
    list_parser.add_argument("--all", action="store_true", help="include completed items")

    done_parser = subparsers.add_parser("done", help="mark a todo as done")
    done_parser.add_argument("id", type=int, help="todo id")

    return parser


def main_problem_9(argv=None):
    parser = build_parser_problem_9()
    args = parser.parse_args(argv)

    if args.command == "add":
        print(f"would add todo: {args.text}")
    elif args.command == "list":
        print(f"would list todos; include_completed={args.all}")
    elif args.command == "done":
        print(f"would mark todo #{args.id} as done")
    else:
        parser.error(f"unknown command: {args.command}")

    return 0


for argv in [
    ["add", "Buy milk"],
    ["list"],
    ["list", "--all"],
    ["done", "2"],
    []
]:
    print("argv:", argv)
    show_result(run_cli(main_problem_9, argv))
    print()

argv: ['add', 'Buy milk']
exit code: 0
--- stdout ---
would add todo: Buy milk


argv: ['list']
exit code: 0
--- stdout ---
would list todos; include_completed=False


argv: ['list', '--all']
exit code: 0
--- stdout ---
would list todos; include_completed=True


argv: ['done', '2']
exit code: 0
--- stdout ---
would mark todo #2 as done


argv: []
exit code: 2
--- stderr ---
usage: todo [-h] {add,list,done} ...
todo: error: the following arguments are required: command




### Best-practice notes

Use subcommands when a program has several distinct actions.

Examples:

- `git commit`, `git push`, `git status`
- `pip install`, `pip list`, `pip uninstall`
- `docker build`, `docker run`, `docker ps`

## Problem 10 — Dispatch subcommands to functions

Improve the previous solution by attaching a function to each subcommand.

Use:

```python
parser.set_defaults(func=some_function)
```

Then `main()` should simply call:

```python
return args.func(args)
```

In [12]:
def handle_add(args):
    print(f"would add todo: {args.text}")
    return 0


def handle_list(args):
    print(f"would list todos; include_completed={args.all}")
    return 0


def handle_done(args):
    print(f"would mark todo #{args.id} as done")
    return 0


def build_parser_problem_10():
    parser = argparse.ArgumentParser(prog="todo-dispatch")
    subparsers = parser.add_subparsers(dest="command", required=True)

    add_parser = subparsers.add_parser("add")
    add_parser.add_argument("text")
    add_parser.set_defaults(func=handle_add)

    list_parser = subparsers.add_parser("list")
    list_parser.add_argument("--all", action="store_true")
    list_parser.set_defaults(func=handle_list)

    done_parser = subparsers.add_parser("done")
    done_parser.add_argument("id", type=int)
    done_parser.set_defaults(func=handle_done)

    return parser


def main_problem_10(argv=None):
    parser = build_parser_problem_10()
    args = parser.parse_args(argv)
    return args.func(args)


for argv in [["add", "Study argparse"], ["list", "--all"], ["done", "7"]]:
    print("argv:", argv)
    show_result(run_cli(main_problem_10, argv))
    print()

argv: ['add', 'Study argparse']
exit code: 0
--- stdout ---
would add todo: Study argparse


argv: ['list', '--all']
exit code: 0
--- stdout ---
would list todos; include_completed=True


argv: ['done', '7']
exit code: 0
--- stdout ---
would mark todo #7 as done




### Best-practice notes

Function dispatch scales better than a long `if` / `elif` chain.

Each subcommand owns its own arguments and its own behavior.

## Problem 11 — Advanced file processor with mode and format groups

Create a CLI named `numbers-file` that reads numbers from a text file.

Arguments:

- positional `path`
- mutually exclusive operation group:
  - `--sum`
  - `--mean`
  - `--median`
- mutually exclusive output format group:
  - `--text`
  - `--json`
  - `--csv`

Defaults:

- operation: `mean`
- format: `text`

The input file contains one number per line.

In [13]:
def read_numbers_file(path):
    numbers = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        stripped = line.strip()
        if not stripped:
            continue
        try:
            numbers.append(float(stripped))
        except ValueError as exc:
            raise ValueError(f"line {line_number}: not a number: {stripped!r}") from exc
    return numbers


def compute_operation(numbers, operation):
    if not numbers:
        raise ValueError("input file contains no numbers")

    if operation == "sum":
        return sum(numbers)
    if operation == "mean":
        return statistics.mean(numbers)
    if operation == "median":
        return statistics.median(numbers)

    raise ValueError(f"unknown operation: {operation}")


def build_parser_problem_11():
    parser = argparse.ArgumentParser(prog="numbers-file")
    parser.add_argument("path", type=existing_file_path)

    op_group = parser.add_mutually_exclusive_group()
    op_group.add_argument("--sum", dest="operation", action="store_const", const="sum")
    op_group.add_argument("--mean", dest="operation", action="store_const", const="mean")
    op_group.add_argument("--median", dest="operation", action="store_const", const="median")

    format_group = parser.add_mutually_exclusive_group()
    format_group.add_argument("--text", dest="format", action="store_const", const="text")
    format_group.add_argument("--json", dest="format", action="store_const", const="json")
    format_group.add_argument("--csv", dest="format", action="store_const", const="csv")

    parser.set_defaults(operation="mean", format="text")
    return parser


def main_problem_11(argv=None):
    parser = build_parser_problem_11()
    args = parser.parse_args(argv)

    try:
        numbers = read_numbers_file(args.path)
        value = compute_operation(numbers, args.operation)
    except ValueError as exc:
        parser.error(str(exc))

    if args.format == "json":
        print(json.dumps({
            "operation": args.operation,
            "value": value,
            "count": len(numbers)
        }, sort_keys=True))
    elif args.format == "csv":
        print("operation,value,count")
        print(f"{args.operation},{value},{len(numbers)}")
    else:
        print(f"{args.operation}: {value}")

    return 0


with tempfile.TemporaryDirectory() as tmpdir:
    path = pathlib.Path(tmpdir) / "numbers.txt"
    path.write_text("10\n20\n30\n", encoding="utf-8")

    for argv in [
        [str(path)],
        [str(path), "--sum"],
        [str(path), "--median", "--json"],
        [str(path), "--sum", "--mean"],
        [str(path), "--json", "--csv"]
    ]:
        print("argv:", argv)
        show_result(run_cli(main_problem_11, argv))
        print()

argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmpcihttrvy\\numbers.txt']
exit code: 0
--- stdout ---
mean: 20.0


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmpcihttrvy\\numbers.txt', '--sum']
exit code: 0
--- stdout ---
sum: 60.0


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmpcihttrvy\\numbers.txt', '--median', '--json']
exit code: 0
--- stdout ---
{"count": 3, "operation": "median", "value": 20.0}


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmpcihttrvy\\numbers.txt', '--sum', '--mean']
exit code: 2
--- stderr ---
usage: numbers-file [-h] [--sum | --mean | --median] [--text | --json | --csv]
                    path
numbers-file: error: argument --mean: not allowed with argument --sum


argv: ['C:\\Users\\user1\\AppData\\Local\\Temp\\tmpcihttrvy\\numbers.txt', '--json', '--csv']
exit code: 2
--- stderr ---
usage: numbers-file [-h] [--sum | --mean | --median] [--text | --json | --csv]
                    path
numbers-file: error: argument --csv: not allowed with argu

### Best-practice notes

This problem separates four responsibilities:

1. parsing arguments
2. reading input
3. computing results
4. formatting output

That separation makes the program easier to test.

## Problem 12 — Build a mini production-style CLI

Create a CLI named `dataset` with subcommands:

```bash
dataset inspect data.csv
dataset stats data.csv --column age
dataset head data.csv --n 5 --json
```

Requirements:

- Use subcommands.
- Use `pathlib.Path`.
- Use custom validation for positive integers.
- Use `--json` for structured output.
- Use function dispatch.

In [14]:
def positive_int(text):
    try:
        value = int(text)
    except ValueError as exc:
        raise argparse.ArgumentTypeError(f"{text!r} is not an integer") from exc

    if value <= 0:
        raise argparse.ArgumentTypeError("value must be a positive integer")

    return value


def read_csv_rows(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def handle_inspect(args):
    rows = read_csv_rows(args.path)
    columns = list(rows[0].keys()) if rows else []
    result = {
        "rows": len(rows),
        "columns": columns,
        "column_count": len(columns),
    }

    if args.json:
        print(json.dumps(result, sort_keys=True))
    else:
        print(f"rows: {result['rows']}")
        print(f"columns: {', '.join(columns)}")

    return 0


def handle_head(args):
    rows = read_csv_rows(args.path)
    selected = rows[:args.n]

    if args.json:
        print(json.dumps(selected, sort_keys=True))
    else:
        for row in selected:
            print(row)

    return 0


def handle_stats(args):
    rows = read_csv_rows(args.path)
    if not rows:
        raise SystemExit("no rows found")

    if args.column not in rows[0]:
        raise SystemExit(f"unknown column: {args.column}")

    values = []
    for index, row in enumerate(rows, start=2):
        try:
            values.append(float(row[args.column]))
        except ValueError:
            raise SystemExit(f"row {index}: column {args.column!r} is not numeric")

    result = {
        "column": args.column,
        "count": len(values),
        "mean": statistics.mean(values),
        "min": min(values),
        "max": max(values),
    }

    if args.json:
        print(json.dumps(result, sort_keys=True))
    else:
        for key, value in result.items():
            print(f"{key}: {value}")

    return 0


def add_common_dataset_args(subparser):
    subparser.add_argument("path", type=existing_file_path)
    subparser.add_argument("--json", action="store_true")


def build_parser_problem_12():
    parser = argparse.ArgumentParser(prog="dataset")
    subparsers = parser.add_subparsers(dest="command", required=True)

    inspect_parser = subparsers.add_parser("inspect", help="inspect a CSV file")
    add_common_dataset_args(inspect_parser)
    inspect_parser.set_defaults(func=handle_inspect)

    head_parser = subparsers.add_parser("head", help="show first rows of a CSV file")
    add_common_dataset_args(head_parser)
    head_parser.add_argument("--n", type=positive_int, default=5)
    head_parser.set_defaults(func=handle_head)

    stats_parser = subparsers.add_parser("stats", help="show numeric stats for a column")
    add_common_dataset_args(stats_parser)
    stats_parser.add_argument("--column", required=True)
    stats_parser.set_defaults(func=handle_stats)

    return parser


def main_problem_12(argv=None):
    parser = build_parser_problem_12()
    args = parser.parse_args(argv)
    return args.func(args)


with tempfile.TemporaryDirectory() as tmpdir:
    path = pathlib.Path(tmpdir) / "people.csv"
    path.write_text(
        "name,age,score\nAda,36,98\nGrace,40,95\nLinus,28,88\n",
        encoding="utf-8"
    )

    for argv in [
        ["inspect", str(path)],
        ["inspect", str(path), "--json"],
        ["head", str(path), "--n", "2"],
        ["stats", str(path), "--column", "age"],
        ["stats", str(path), "--column", "score", "--json"],
        ["head", str(path), "--n", "0"]
    ]:
        print("argv:", argv)
        show_result(run_cli(main_problem_12, argv))
        print()

argv: ['inspect', 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmpyo18uxrf\\people.csv']
exit code: 0
--- stdout ---
rows: 3
columns: name, age, score


argv: ['inspect', 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmpyo18uxrf\\people.csv', '--json']
exit code: 0
--- stdout ---
{"column_count": 3, "columns": ["name", "age", "score"], "rows": 3}


argv: ['head', 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmpyo18uxrf\\people.csv', '--n', '2']
exit code: 0
--- stdout ---
{'name': 'Ada', 'age': '36', 'score': '98'}
{'name': 'Grace', 'age': '40', 'score': '95'}


argv: ['stats', 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmpyo18uxrf\\people.csv', '--column', 'age']
exit code: 0
--- stdout ---
column: age
count: 3
mean: 34.666666666666664
min: 28.0
max: 40.0


argv: ['stats', 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmpyo18uxrf\\people.csv', '--column', 'score', '--json']
exit code: 0
--- stdout ---
{"column": "score", "count": 3, "max": 98.0, "mean": 93.66666666666667, "min": 88.0}


argv: ['head', 'C

### Best-practice notes

Production-style CLIs benefit from:

- small handler functions
- reusable argument helpers
- clear validation
- structured output for automation
- consistent exit codes
- tests that call `main(argv)` directly

## Final challenge — Design your own advanced CLI

Design a CLI with:

1. at least one positional argument
2. at least one optional argument
3. at least one boolean flag
4. one mutually exclusive group
5. one custom type validator
6. at least two subcommands
7. a `main(argv=None)` function
8. at least five notebook tests using `run_cli()`

Suggested ideas:

- `image-tool resize|inspect`
- `gradebook add|stats`
- `finance convert|summarize`
- `notes add|search|list`
- `backup plan|run`